## 功能介绍：识别用人物图片，响应人物信息及其最新资讯

In [1]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from rich import print as rprint

load_dotenv()

model = init_chat_model(
    model="qwen3-vl-plus",  # 必须支持 多模态 + Function calling
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

web_search = TavilySearch(
    max_results=10,
    topic="news",
    search_depth="basic",
    # 注意：time_range 和 start_date/end_date 互斥，这里用 start_date/end_date 控制范围
)

In [2]:
from langchain.agents.middleware import AgentMiddleware, hook_config
from langchain_core.messages import AIMessage, SystemMessage, ToolMessage


# 阶段1 专用：精简识别 prompt，强制模型只做识别不做搜索
RECOGNITION_PROMPT = """你是一个人物图像识别助手。分析图片中的人物，仅输出置信度最高的一位人物的全名。不要输出任何其他内容。"""

class TwoPhaseRecognitionMiddleware(AgentMiddleware):
    """两阶段人物识别中间件
    
    阶段1（识别）：剥离所有工具，强制模型只能输出纯文本人物名
    阶段2（搜索）：恢复工具，模型自行根据 system_prompt 搜索并整合输出
    """

    def wrap_model_call(self, request, handler):
        messages = request.state.get("messages", [])

        # 判断当前处于哪个阶段
        has_recognition = any(
            isinstance(m, AIMessage) and m.content and not getattr(m, "tool_calls", None)
            for m in messages
        )
        has_tool_result = any(isinstance(m, ToolMessage) for m in messages)

        if not has_recognition and not has_tool_result:
            # 阶段1：还没识别 → 剥离工具，强制纯文本输出
            return handler(request.override(
                tools=[],
                system_message=SystemMessage(content=RECOGNITION_PROMPT)
            ))

        # 阶段2/3：已识别或已搜索 → 恢复正常工具
        return handler(request)

    @hook_config(can_jump_to=["model"])
    def after_model(self, state, runtime):
        messages = state["messages"]
        last_msg = messages[-1] if messages else None

        # 只处理 AIMessage
        if not isinstance(last_msg, AIMessage):
            return None
        # 正在调工具 → 不干预
        if getattr(last_msg, "tool_calls", None):
            return None
        # 已有搜索结果 → 模型正在输出最终答案，不干预
        if any(isinstance(m, ToolMessage) for m in messages):
            return None

        # 识别刚完成 → 拉回 model，tools 已恢复，模型自行搜索
        name = (last_msg.content or "").strip()
        if not name or "未知" in name:
            return None

        return {"jump_to": "model"}

In [3]:
from datetime import datetime

today = datetime.now().strftime("%Y年%m月%d日")
today_raw = datetime.now().strftime("%Y-%m-%d")
this_year = datetime.now().strftime("%Y")

system_prompt = f"""
你是一个人物图像识别与信息整合助手。收到用户提供的人物照片后，严格按以下三步处理：

**重要：当前日期是 {today}。**

步骤1 — 视觉识别
分析图片中的人物，仅输出置信度最高的一位人物的全名。

步骤2 — 信息检索
基于步骤1识别出的姓名，调用 web_search 工具搜索该人物的最近的新闻。
**搜索要求：**
- 必须传入 start_date="{this_year}-01-01"，end_date="{today_raw}"，确保只获取今年的结果

步骤3 — 结构化输出
整合检索结果，按以下格式输出：
 - 姓名：
 - 基本信息（包括出生日期、职业/身份、主要成就/代表作）：
 - 近期动态：
 - 数据来源：标题:链接

规则：
 - 必须使用 web_search 工具，信息不足时明确标注"未公开"或"不详"
 - 禁止编造任何细节，所有信息必须有可追溯来源
 - 你必须使用英文构造搜索 query，以获得更准确的搜索结果

"""

In [4]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    middleware=[TwoPhaseRecognitionMiddleware()],
)

In [9]:
human_message = HumanMessage(
    content=[
        {
            "type": "image_url",
            "image_url": {
                "url": "https://www.18fxzy.com/upload/products/202110/02/1032266157c4ba42ce39fHhp9.jpg"}
        }
    ]
)

In [10]:
resp = agent.invoke({"messages": [human_message]})

In [11]:
rprint(resp)

{
    'messages': [
        HumanMessage(
            content=[
                {
                    'type': 'image_url',
                    'image_url': {
                        'url': 'https://www.18fxzy.com/upload/products/202110/02/1032266157c4ba42ce39fHhp9.jpg'
                    }
                }
            ],
            additional_kwargs={},
            response_metadata={},
            id='3b2e5560-4deb-4ed5-ae6f-33d7b08a8b35'
        ),
        AIMessage(
            content='周淑怡',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 4,
                    'prompt_tokens': 386,
                    'total_tokens': 390,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': None,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 4
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': 342,
                        'text_tokens': 44
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3-vl-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-7cb6e71f-d640-9d75-8d8d-5f4e3b93064b',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f22fb-953a-7c70-9db8-a4a939ad0356-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 386,
                'output_tokens': 4,
                'total_tokens': 390,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 72,
                    'prompt_tokens': 2411,
                    'total_tokens': 2483,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': None,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 72
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': 342,
                        'text_tokens': 2069
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3-vl-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-189f2428-6711-9bd7-9bed-2c63a5309bc3',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f22fb-f440-7730-a457-43b9a089fb6e-0',
            tool_calls=[
                {
                    'name': 'tavily_search',
                    'args': {
                        'query': 'Zhou Shuyi recent news 2026',
                        'start_date': '2026-01-01',
                        'end_date': '2026-07-02'
                    },
                    'id': 'call_d07c5202e0014ef7a0e09f95',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 2411,
                'output_tokens': 72,
                'total_tokens': 2483,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
    

In [12]:
for message in resp['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image_url', 'image_url': {'url': 'https://www.18fxzy.com/upload/products/202110/02/1032266157c4ba42ce39fHhp9.jpg'}}]
================================== Ai Message ==================================

周淑怡
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_d07c5202e0014ef7a0e09f95)
 Call ID: call_d07c5202e0014ef7a0e09f95
  Args:
    query: Zhou Shuyi recent news 2026
    start_date: 2026-01-01
    end_date: 2026-07-02
================================= Tool Message =================================
Name: tavily_search

{"query": "Zhou Shuyi recent news 2026", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.formula1.com/en/latest/article/exclusive-lowdon-on-why-zhou-ticked-all-of-the-boxes-as-a-reserve-for.5st4cPCtlVdW5VVPbfCzR0", "title": "Lowdon on why Zhou ‘ticked all the boxes’ for Cadilla

# 5.踩坑记录与经验总结

## 问题 1：Agent 不调用 Tool（tool_calls 始终为空）

**现象**：Agent 直接输出完整回答，数据来源链接全是编造的。

**根因**：`qwen-vl-max` 不支持 Function Calling。Agent 把工具绑到了模型上，但 API 请求中的 `tools` 参数被模型忽略，直接返回了文本。

**解决**：换成同时支持多模态 + Function Calling 的模型 `qwen3-vl-plus`。

> **教训**：选模型时必须确认三个能力——视觉理解、工具调用、指令遵循，缺一不可。

---

## 问题 2：搜索时间范围偏旧（2023-2024）

**现象**：联网搜索的结果全是 2023-2024 年的过时内容。

**根因**：模型的训练数据截止于 2024 年中，它"以为"现在是 2024 年，搜索时传了 `start_date: 2023-01-01, end_date: 2024-06-01`。

**解决**：在 system prompt 中动态注入当前日期：
```python
from datetime import datetime
today = datetime.now().strftime("%Y年%m月%d日")
```

> **教训**：模型没有实时时钟，需要外部注入时间锚点。

---

## 问题 3：time_range 与 start_date/end_date 互斥（HTTP 400）

**现象**：`Error 400: When time_range is set, start_date or end_date cannot be set`

**根因**：Tavily API 规定 `time_range` 和 `start_date`/`end_date` 只能二选一。初始化设了 `time_range="year"`，prompt 又要求模型传 `start_date`/`end_date` → 冲突。

**解决**：去掉初始化时的 `time_range`，仅通过 prompt 控制模型传 `start_date`/`end_date`。

> **教训**：初始化参数和调用时参数可能冲突，优先搞清楚 API 的互斥规则。

---

## 问题 4：搜索结果偏 2025 年，2026 年内容少

**现象**：搜索到的结果以 2025 年为主，2026 年只有零星几条。

**根因**：三重叠加——
- 模型构造的 query 偏百科（`birth date`、`career highlights`），不是新闻 query
- Tavily 按**相关性**排序非时间排序，深度报道压过碎片新闻
- 模型没传 `start_date` 限定今年

**解决**：prompt 中硬约束：
```
- 必须传入 start_date="2026-01-01"，end_date="2026-07-01"
- query 必须包含 "2026" 年份关键词
```

> **教训**：搜索引擎默认按相关性排序，需要精确的时间过滤参数才能拿到最新内容。

---

## TavilySearch 参数速查

### `topic` — 搜索类别

| 值 | 适用场景 |
|----|---------|
| `general` | 默认，绝大多数查询 |
| `news` | 主流媒体报道的政治、体育等重大事件 |
| `finance` | 市场、投资、经济数据、财经新闻 |

> `news` 不是"搜新东西"的意思——只是限定新闻源。日常搜"最新 XX"用 `general` 即可。

### `search_depth` — 搜索深度

| 值 | 延迟 | 特点 |
|----|------|------|
| `ultra-fast` | ~0.3s | 最快，适合实时提示/补全 |
| `fast` | ~0.5s | 快 + 高相关，适合简单事实查询 |
| `basic` | ~1s | 默认，平衡速度与覆盖面，日常够用 |
| `advanced` | ~2-5s | 最深，适合冷门话题/需要多源验证 |

> `basic` 和 `advanced` 对大部分查询差别不大，`advanced` 仅在搜冷门/专业信息时有额外召回。日常用 `basic` 更省 API 消耗。

### 参数分类

| 类型 | 参数 | 说明 |
|------|------|------|
| 仅初始化 | `max_results`, `include_answer`, `include_raw_content`, `include_images`, `country`, `exact_match`, `auto_parameters` | 构造时固定，模型改不了 |
| 可被覆盖 | `topic`, `search_depth`, `time_range` | 初始化设了优先，不设则模型可传 |
| 仅调用时 | `query`, `start_date`, `end_date`, `include_domains`, `exclude_domains` | 只能模型传 |

---

## 最终可用配置

```python
# 模型：多模态 + Function Calling 缺一不可
model = init_chat_model(model="qwen3-vl-plus")

# 搜索：time_range 和 start_date/end_date 二选一
web_search = TavilySearch(max_results=5, topic="news", search_depth="basic")

# Prompt：注入时间 + 硬约束搜索范围
system_prompt = f"""
当前日期：{datetime.now().strftime("%Y年%m月%d日")}
搜索时必须传 start_date="{this_year}-01-01", end_date="{today_raw}"
query 必须包含 "{this_year}" 年份
"""
```

# 6.Agent 工作流深入理解

## Agent 如何决定"继续"还是"停止"？

### 本质：不是判断，是预测

大模型不"判断"结果是否满足要求，它只做一件事——**预测下一个 token**。

```
Agent 循环的每一步：
  messages = [System, Human, AI(tool_call), Tool(结果), ...]
  response = model.invoke(messages)

  response.tool_calls 非空？→ 继续循环，执行工具，结果回传
  response.tool_calls 为空？→ 停止，输出文本作为最终答案
```

**传统程序**：`if result != good: retry()`

**大模型**：`P(next_token = tool_call | messages) > P(next_token = text)` 就继续，否则停止。

没有 if 判断，只有概率分布。所谓的"判断够了"，只是训练数据中"拿到足够信息后输出答案"这个模式被学得足够好。

### 分工

| 谁做的 | 干什么 |
|--------|------|
| **LLM** | 输入消息列表 → 预测输出（文本 或 tool_call） |
| **Agent 框架** | 读 `tool_calls` 字段 → 有则执行工具回传 / 无则结束 |
| **工具** | 执行实际动作（搜索、API 调用等），返回结果 |

Agent 从不给结果打分，从不检查质量。它只问模型一个问题：**"你还想调工具吗？"**

---

## 核心问题：模型觉得够了，但实际不够

### 典型场景

```
模型搜了一次 → 返回3条结果 → 模型："够了" → 输出文本
但你发现：缺少出生日期、动态只有2年前的、来源链接不完整
```

**模型提前"下班"了。** 根因是停止决策权完全在模型手里，而模型的"够了"标准不一定等于你的标准。

### 三层防线

```
┌─────────────────────────────────────────────┐
│ ① Prompt 约束层    ← 先试试，改动最小         │
│ ② Middleware 拦截层 ← 硬规则，代码兜底         │
│ ③ QA Agent 审查层  ← 最高保障，另一个模型把关   │
└─────────────────────────────────────────────┘
```

---

## ① Prompt 约束：把标准写死

把"够了"的定义从模糊变精确：

```python
# ❌ 模糊 —— 模型自己解释"够"
"搜索该人物的信息并输出"

# ✅ 硬标准 —— 不达标就不能停
"""输出前必须逐项自查，以下字段全部填写且有可追溯来源才算完成：
- 姓名、出生日期、出生地、职业、代表作、近期动态
- 任一字段缺失 → 必须重新搜索，不得输出最终结果"""
```

**适用场景**：偏差不大的情况，改动成本最低。

---

## ② Middleware 拦截：输出前硬检查

Prompt 是建议，Middleware 是规则。模型说停了，中间件在输出前拦截并强制拉回循环：

```python
from langchain.agents.middleware import AgentMiddleware

class QualityCheckMiddleware(AgentMiddleware):

    def before_model(self, state, runtime):
        messages = state["messages"]
        last_msg = messages[-1] if messages else None
        if last_msg is None:
            return None

        # 如果是最终文本输出（非 tool_call），检查是否达标
        if hasattr(last_msg, "content") and last_msg.content \
           and not getattr(last_msg, "tool_calls", None):
            required = ["出生日期", "近期动态", "数据来源"]
            missing = [f for f in required
                       if f not in str(last_msg.content)]

            if missing:
                # 不结束！强制追加指令继续搜索
                return {"messages": [HumanMessage(
                    content=f"以下字段仍缺失: {', '.join(missing)}。"
                            f"请重新调用 web_search 搜索补充。"
                )]}

        return None
```

**适用场景**：有硬性字段要求、需要代码级兜底保障。

---

## ③ QA Agent 审查：双模型把关

执行 Agent 干活，审查 Agent 验收。不合格就把审查意见喂回执行 Agent：

```python
# 执行 Agent —— 负责搜索和输出
executor = create_agent(model=model, tools=[web_search])

# 审查 Agent —— 用纯文本模型，不需要工具
reviewer = create_agent(
    model=model,
    system_prompt="""你是质量审查员，检查输出是否符合标准：
    1. 每个字段都有搜索结果支撑（非推测）
    2. 每条信息来源都可追溯
    不合格 → 以"REJECT: <缺失项>"开头
    合格   → 以"PASS"开头"""
)

# 循环：执行 → 审查 → 不通过就回炉
for i in range(max_rounds := 3):
    result = executor.invoke(...)
    review = reviewer.invoke({"messages": [HumanMessage(
        f"审查以下输出：\n{result['messages'][-1].content}"
    )]})

    if "PASS" in review["messages"][-1].content:
        break
    # REJECT → 把审查意见喂回 executor，触发重新搜索
```

**适用场景**：高可靠性要求、允许额外的 token 和时间开销。

---

## 总结

| 维度 | 说明 |
|------|------|
| **谁决定停** | LLM 通过是否输出 `tool_calls` 决定 |
| **Agent 做什么** | 只读 `tool_calls` 字段，有就执行，没有就停 |
| **核心风险** | 模型的"够了"≠你的"够了" |
| **解法** | 把停止标准从模型手里收回来 → Prompt 写死 / Middleware 拦截 / QA Agent 审查 |
| **推荐路径** | 先方案①（Prompt），不够加方案②（Middleware），高可靠场景上方案③ |

# 7.Middleware 两阶段方案：让识别结果显式可见

## 问题背景

改造前的 Agent 在第一轮 API 调用中，模型同时完成识别和搜索决策，输出为：

```
AIMessage(content='', tool_calls=[tavily_search("林允 2026 latest news")])
```

**人名被隐含在 `tool_calls` 的 query 参数里，`content` 为空。** 这不是 bug——当模型同时看到图片和 tools 时，它会走最短概率路径：直接调工具，跳过输出人名的文本步骤。

如果需要在 `content` 中显式拿到识别出的人名（用于日志、审计、展示），就需要在"识别"和"搜索"之间加入一道模型跳不过去的**硬边界**。

---

## 方案演进

| 版本 | 做法 | 消息流 | API 调用 |
|------|------|--------|----------|
| 改造前 | 无中间件，模型自主 | `Human → AI(tool_call) → Tool → AI(最终)` | 2 次 |
| 初版 MW | after_model 注入搜索指令 | `Human → AI("人名") → Human(注入) → AI(tool_call) → Tool → AI(最终)` | 3 次 |
| **最终版** | **只控制 tools 可见性** | `Human → AI("人名") → AI(tool_call) → Tool → AI(最终)` | **3 次** |

最终版删掉了 `after_model` 中注入 `HumanMessage` 的逻辑，因为注入不仅增加了冗余消息，还把"搜索什么、怎么搜"的决策权从模型手中转移到了中间件代码中。最终版的思路是：**Middleware 只管 tools 可见性这道门，门开了之后模型自己决定怎么走。**

---

## 工作原理

### 两个钩子的分工

| | `wrap_model_call` | `after_model` |
|---|---|---|
| **时机** | 每次调用 API **之前** | 模型每次返回 **之后** |
| **职责** | 改 `request`（控制 tools/prompt） | 改 `state`（控制 jump_to） |
| **阶段1** | `tools=[]`，关闭，模型只能输出文本 | `jump_to="model"`，拉回循环 |
| **阶段2** | tools 恢复，正常放行 | 不干预 |
| **阶段3** | 放行 | 不干预 |

### 执行时序

```
agent.invoke({"messages": [HumanMessage(image)]})
  │
  ▼
┌─ 第1轮 ─────────────────────────────────────────────────────┐
│  wrap_model_call: tools=[] + 精简识别 prompt                 │
│       → API 调用①: 模型看到图片，没有工具可选                  │
│       → AIMessage(content="埃马纽埃尔·马克龙")  ← 纯文本！    │
│                                                              │
│  after_model: 识别完成，tool_calls 为空，循环即将终止          │
│       → return {"jump_to": "model"}  ← 拉回来！              │
└──────────────────────────────────────────────────────────────┘
  │
  ▼
┌─ 第2轮 ─────────────────────────────────────────────────────┐
│  wrap_model_call: tools=[tavily_search] ← 工具已恢复！       │
│       → API 调用②: 模型看到人名 + tools + system_prompt      │
│       → AIMessage(tool_calls=[tavily_search("Emmanuel Macron 2026")]) │
│                                                              │
│  after_model: 正在调工具 → 不干预                              │
└──────────────────────────────────────────────────────────────┘
  │
  ▼ ToolNode 执行 TavilySearch
  │
  ▼
┌─ 第3轮 ─────────────────────────────────────────────────────┐
│  wrap_model_call: tools 正常，放行                            │
│       → API 调用③: 模型看到搜索结果，整合输出                 │
│       → AIMessage(content="姓名：...\n基本信息：...")         │
│                                                              │
│  after_model: 已有 ToolMessage → 不干预 → 正常结束            │
└──────────────────────────────────────────────────────────────┘
```

---

## 关键代码

```python
  wrap_model_call

  触发时机：每次要调模型 API 之前

  它做的事：控制模型"能看到什么"

  def wrap_model_call(self, request, handler):
      # 检查状态：是否已完成识别、是否已有搜索结果
      if 还没识别过 且 还没搜过:
          # 阶段1 → 拿掉 tools + 换成精简识别 prompt
          return handler(request.override(tools=[],
  system_message=RECOGNITION_PROMPT))
      # 阶段2/3 → 原样放行，tools 恢复正常
      return handler(request)

  用一句话说：阶段1 把 tools 藏起来，阶段2 把 tools 还回去。
  这是实现"硬边界"的核心——模型在第一轮看不到工具，不可能走捷径。

  ---
  after_model

  触发时机：每次模型返回结果之后

  它做的事：防止 Agent 提前"下班"

  def after_model(self, state, runtime):
      # 不是 AIMessage → 不干预
      # AIMessage 有 tool_calls → 模型在调工具，不干预
      # 已有 ToolMessage → 已经搜索过了，模型在输出最终答案，不干预
      # 识别出的人名是"未知" → 没法搜索，不干预

      # 到此：识别刚完成，但 tools 没了，tool_calls 为空，循环会停
      return {"jump_to": "model"}  # ← 拉回来，进入阶段2

  正常 Agent 循环的停止条件是 AIMessage.tool_calls 为空 → 路由到 END。阶段1
  识别完成后工具被拿掉了，tool_calls 必然为空，Agent 会直接结束。

  after_model
  的作用就在这个时间窗口：识别人名已经拿到、工具还没恢复、循环即将终止——赶紧
  jump_to="model" 把它拽回来，让 wrap_model_call 在下一次执行时把 tools 还回去。
```

---

## 效果验证

```python
# 改造前
[0] HumanMessage:
[1] AIMessage: content=''  tool_calls=[tavily_search(...)]  ← 人名不可见

# 改造后
[0] HumanMessage:
[1] AIMessage: content=埃马纽埃尔·马克龙                    ← 人名显式可见
[2] AIMessage: tool_calls=[tavily_search("Emmanuel Macron 2026")]
[3] ToolMessage:
[4] AIMessage: content=- 姓名：埃马纽埃尔·马克龙...
```

---

## 设计原则总结

1. **Middleware 只管硬边界，不管决策。** 它只控制 tools 的可见性，不替模型决定"应该搜什么"。门开了，模型自己走。

2. **Prompt 是建议，Middleware 是规则。** system_prompt 写"先识别再搜索"只是概率建议；`tools=[]` 才是不可逾越的硬约束。

3. **每个钩子只做一件事。** `wrap_model_call` 改 request，`after_model` 改 state。职责不重叠，边界清晰。

4. **最小干预原则。** 除了阶段1强制关闭 tools、阶段1结束后拉回循环，其余时刻完全不插手。模型在阶段2和阶段3的行为完全由原始 system_prompt 驱动。